# 17 — Recomendação de ação para revisão humana

Este módulo não executa ações sobre clientes. Ele converte os indicadores em uma label de próximo passo e marca `revisao_humana=True` em todos os casos.


## Confiança comparável somente dentro de cada mecanismo

Como scores de modelo e heurística têm naturezas diferentes, o limiar serve apenas para direcionar casos frágeis à revisão manual; ele não é apresentado como probabilidade calibrada.

O limiar de 0,55 encaminha evidências fracas para revisão. Scores de produtos, modelos e regras têm significados diferentes e não devem ser comparados como probabilidades iguais.


In [ ]:
def _has_low_confidence(
    products: list[dict[str, Any]],
    sentiment: dict[str, Any],
    churn: dict[str, Any],
    opportunity: dict[str, Any],
) -> bool:
    indicators = (sentiment, churn, opportunity)
    if any(
        indicator["score_type"] == "model_probability"
        and float(indicator["score"]) < 0.55
        for indicator in indicators
    ):
        return True

    active_heuristic_scores = []
    if sentiment["score_type"] == "heuristic" and sentiment["label"] not in {"neutro", "misto"}:
        active_heuristic_scores.append(float(sentiment["score"]))
    if churn["score_type"] == "heuristic" and churn["label"] != "baixo":
        active_heuristic_scores.append(float(churn["score"]))
    if opportunity["score_type"] == "heuristic" and opportunity["label"] == "detectada":
        active_heuristic_scores.append(float(opportunity["score"]))
    if any(score < 0.55 for score in active_heuristic_scores):
        return True

    return bool(
        opportunity["label"] == "detectada"
        and products
        and float(products[0]["score"]) < 0.65
    )


## Ordem de prioridade

Churn alto aciona retenção. Depois vêm revisão manual, demonstração com produto explicitamente fundamentado, qualificação sem produto confiável e, por fim, acompanhamento da conta.

A prioridade é aplicada exatamente nesta sequência. A demonstração só aparece quando há oportunidade, produto explícito, fonte e score suficiente; toda label ainda exige revisão humana.


In [ ]:
def _recommend_action(
    products: list[dict[str, Any]],
    sentiment: dict[str, Any],
    churn: dict[str, Any],
    opportunity: dict[str, Any],
) -> dict[str, Any]:
    if churn["label"] == "alto":
        label = "acionar_retencao"
    elif sentiment["label"] == "misto" or _has_low_confidence(
        products, sentiment, churn, opportunity
    ):
        label = "revisar_manualmente"
    elif opportunity["label"] == "detectada":
        product_is_grounded = bool(
            products
            and products[0]["score"] >= 0.65
            and products[0]["explicit_match"]
            and products[0]["sources"]
        )
        label = (
            "agendar_demonstracao"
            if product_is_grounded
            else "qualificar_oportunidade"
        )
    else:
        label = "acompanhar_conta"
    return {"label": label, "revisao_humana": True}
